In [5]:
import polars as pl
from pathlib import Path

class DataPipeline:
    def __init__(self, input_path: str, output_path: str):
        self.input_path = Path(input_path)
        self.output_path = Path(output_path)
        self.invalid_values = ["none", "", "nan", "unknown"]
        
        # Define columns to drop
        self.drop_cols = [
            "about", "edu_list", "age", "first_major", "highest_degree",
            "duration_edu_certi", "first_edu_start", "end_edu_first",
            "experience", "duration_exp", "total_experience", "subtitle",
            "title_certi", "update_certi", "second_degree/certi_date",
            "update_duration", "first_certi", "last_certi",
            "last_major/certi_date", "total_time", "credential_url",
            "certi_standard", "certi_standard_stem", "first_industry_standard_1",
            "second_major/certi_standard", "second_major/certi_standard_stem",
            "first_major_standard_1", "last_certi_standard", "last_certi_standard_stem"
        ]

    def _is_valid_expr(self, col_name: str) -> pl.Expr:
        """Helper to check if a column contains meaningful data."""
        return (
            pl.col(col_name).is_not_null() & 
            (pl.col(col_name).str.strip_chars().str.to_lowercase().is_in(self.invalid_values).not_()) &
            (pl.col(col_name).str.strip_chars() != "")
        )

    def run(self):
        """Executes the lazy processing pipeline."""
        try:
            print(f"Initializing pipeline: {self.input_path}")
            
            # Use LazyFrame for memory efficiency
            lf = pl.scan_csv(self.input_path, infer_schema_length=0)

            # 1. Drop unnecessary columns
            lf = lf.drop(self.drop_cols)

            # 2. Define valid flags for each row
            is_temporal_valid = self._is_valid_expr("start_year") | self._is_valid_expr("start_date")
            is_industry_valid = self._is_valid_expr("first_industry") | self._is_valid_expr("current_industry")
            
            # 3. Create a mask: True if the row satisfies BOTH criteria
            lf = lf.with_columns(
                is_valid_row = (is_temporal_valid & is_industry_valid)
            )

            # 4. Window Function: If ANY row for this ID is valid, keep the whole ID
            # This prevents truncation of rows for a valid user.
            lf = lf.filter(
                pl.col("is_valid_row").any().over("id")
            ).drop("is_valid_row")

            print(f"Executing transformations and saving to {self.output_path}...")
            
            # Note: sink_csv does not support all operations (like window functions) 
            # in some Polars versions. If sink_csv fails, use collect().write_csv().
            lf.collect().write_csv(self.output_path)
            
            print("Pipeline execution successful.")

        except Exception as e:
            print(f"Critical Pipeline Error: {e}")

INPUT = "../Data/ha_original_data.csv"
OUTPUT = "../Data/ha_processed_data.csv"

pipeline = DataPipeline(INPUT, OUTPUT)
pipeline.run()

Initializing pipeline: ../Data/ha_original_data.csv
Executing transformations and saving to ../Data/ha_processed_data.csv...
Pipeline execution successful.
